# Aviation Accidents Analysis

You are part of a consulting firm that is tasked to do an analysis of commercial and passenger jet airline safety. The client (an airline/airplane insurer) is interested in knowing what types of aircraft (makes/models) exhibit low rates of total destruction and low likelihood of fatal or serious passenger injuries in the event of an accident. They are also interested in any general variables/conditions that might be at play. Your analysis will be based off of aviation accident data accumulated from the years 1948-2023. 

Our client is only interested in airplane makes/models that are professional builds and could potentially still be active. Assume a max lifetime of 40 years for a make/model retirement and make sure to filter your data accordingly (i.e. from 1983 onwards). They would also like separate recommendations for small aircraft vs. larger passenger models. **In addition, make sure that claims that you make are statistically robust and that you have enough samples when making comparisons between groups.**


In this summative assessment you will demonstrate your ability to:
- **Use Pandas to load, inspect, and clean the dataset appropriately.**
- **Transform relevant columns to create measures that address the problem at hand.**
- conduct EDA: visualization and statistical measures to systematically understand the structure of the data
- recommend a set of airplanes and makes conforming to the client's request and identify at least *two* factors contributing to airplane safety. You must provide supporting evidence (visuals, summary statistics, tables) for each claim you make.

### Make relevant library imports

In [105]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

## Data Loading and Inspection

### Load in data from the relevant directory and inspect the dataframe.
- inspect NaNs, datatypes, and summary statistics

In [106]:
# data loading 
df=pd.read_csv("AviationData.csv",index_col=0, encoding="latin-1")

C:\Users\Lenovo\AppData\Local\Temp\ipykernel_16628\3338872839.py:2: DtypeWarning: Columns (0: Longitude, 1: Airport.Code, 2: Report.Status) have mixed types. Specify dtype option on import or set low_memory=False.
  df=pd.read_csv("AviationData.csv",index_col=0, encoding="latin-1")


In [107]:
#check the shape of the data 
df.shape

(88889, 30)

In [108]:
#displays column names, data types, and non-null counts
df.info()

<class 'pandas.DataFrame'>
Index: 88889 entries, 20001218X45444 to 20221230106513
Data columns (total 30 columns):
 #   Column                  Non-Null Count  Dtype  
---  ------                  --------------  -----  
 0   Investigation.Type      88889 non-null  str    
 1   Accident.Number         88889 non-null  str    
 2   Event.Date              88889 non-null  str    
 3   Location                88837 non-null  str    
 4   Country                 88663 non-null  str    
 5   Latitude                34382 non-null  object 
 6   Longitude               34373 non-null  object 
 7   Airport.Code            50132 non-null  str    
 8   Airport.Name            52704 non-null  str    
 9   Injury.Severity         87889 non-null  str    
 10  Aircraft.damage         85695 non-null  str    
 11  Aircraft.Category       32287 non-null  str    
 12  Registration.Number     87507 non-null  str    
 13  Make                    88826 non-null  str    
 14  Model                   88797 no

In [109]:
#counting missing values in each column
df.isna().sum()

Investigation.Type            0
Accident.Number               0
Event.Date                    0
Location                     52
Country                     226
Latitude                  54507
Longitude                 54516
Airport.Code              38757
Airport.Name              36185
Injury.Severity            1000
Aircraft.damage            3194
Aircraft.Category         56602
Registration.Number        1382
Make                         63
Model                        92
Amateur.Built               102
Number.of.Engines          6084
Engine.Type                7096
FAR.Description           56866
Schedule                  76307
Purpose.of.flight          6192
Air.carrier               72241
Total.Fatal.Injuries      11401
Total.Serious.Injuries    12510
Total.Minor.Injuries      11933
Total.Uninjured            5912
Weather.Condition          4492
Broad.phase.of.flight     27165
Report.Status              6384
Publication.Date          13771
dtype: int64

In [110]:
# get the summary statistics
df.describe()

,Number.of.Engines,Total.Fatal.Injuries,Total.Serious.Injuries,Total.Minor.Injuries,Total.Uninjured
count,82805.000000,77488.000000,76379.000000,76956.000000,82977.000000
mean,1.146585,0.647855,0.279881,0.357061,5.325440
std,0.446510,5.485960,1.544084,2.235625,27.913634
min,0.000000,0.000000,0.000000,0.000000,0.000000
25%,1.000000,0.000000,0.000000,0.000000,0.000000
50%,1.000000,0.000000,0.000000,0.000000,1.000000
75%,1.000000,0.000000,0.000000,0.000000,2.000000
max,8.000000,349.000000,161.000000,380.000000,699.000000


## Data Cleaning

### Filtering aircrafts and events

We want to filter the dataset to include aircraft that the client is interested in an analysis of:
- inspect relevant columns
- figure out any reasonable imputations
- filter the dataset

In [111]:
df.columns

Index(['Investigation.Type', 'Accident.Number', 'Event.Date', 'Location',
       'Country', 'Latitude', 'Longitude', 'Airport.Code', 'Airport.Name',
       'Injury.Severity', 'Aircraft.damage', 'Aircraft.Category',
       'Registration.Number', 'Make', 'Model', 'Amateur.Built',
       'Number.of.Engines', 'Engine.Type', 'FAR.Description', 'Schedule',
       'Purpose.of.flight', 'Air.carrier', 'Total.Fatal.Injuries',
       'Total.Serious.Injuries', 'Total.Minor.Injuries', 'Total.Uninjured',
       'Weather.Condition', 'Broad.phase.of.flight', 'Report.Status',
       'Publication.Date'],
      dtype='str')

In [112]:
#getting the percentage of the missing values and dorting them in descending order
(round(df.isna().sum()/len(df),4)*100).sort_values(ascending=False)

Schedule                  85.85
Air.carrier               81.27
FAR.Description           63.97
Aircraft.Category         63.68
Longitude                 61.33
Latitude                  61.32
Airport.Code              43.60
Airport.Name              40.71
Broad.phase.of.flight     30.56
Publication.Date          15.49
Total.Serious.Injuries    14.07
Total.Minor.Injuries      13.42
Total.Fatal.Injuries      12.83
Engine.Type                7.98
Report.Status              7.18
Purpose.of.flight          6.97
Number.of.Engines          6.84
Total.Uninjured            6.65
Weather.Condition          5.05
Aircraft.damage            3.59
Registration.Number        1.55
Injury.Severity            1.12
Country                    0.25
Amateur.Built              0.11
Model                      0.10
Make                       0.07
Location                   0.06
Accident.Number            0.00
Event.Date                 0.00
Investigation.Type         0.00
dtype: float64

In [113]:
#describe numerical columns
df.describe(include="number")

,Number.of.Engines,Total.Fatal.Injuries,Total.Serious.Injuries,Total.Minor.Injuries,Total.Uninjured
count,82805.000000,77488.000000,76379.000000,76956.000000,82977.000000
mean,1.146585,0.647855,0.279881,0.357061,5.325440
std,0.446510,5.485960,1.544084,2.235625,27.913634
min,0.000000,0.000000,0.000000,0.000000,0.000000
25%,1.000000,0.000000,0.000000,0.000000,0.000000
50%,1.000000,0.000000,0.000000,0.000000,1.000000
75%,1.000000,0.000000,0.000000,0.000000,2.000000
max,8.000000,349.000000,161.000000,380.000000,699.000000


In [114]:
#describe categorical columns
df.describe(include="O").T

C:\Users\Lenovo\AppData\Local\Temp\ipykernel_16628\2177166514.py:2: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  df.describe(include="O").T


,count,unique,top,freq
Investigation.Type,88889,2,Accident,85015
Accident.Number,88889,88863,ERA22LA103,2
Event.Date,88889,14782,1982-05-16,25
Location,88837,27758,"ANCHORAGE, AK",434
Country,88663,219,United States,82248
Latitude,34382,25592,332739N,19
Longitude,34373,27156,0112457W,24
Airport.Code,50132,10374,NONE,1488
Airport.Name,52704,24870,Private,240
Injury.Severity,87889,109,Non-Fatal,67357


In [115]:
# #inspect the relevant columns 
# relevant_columns=[
#                 'Event.Date'
#                 'Accident.Number',
#                   'Make', 
#                   'Model',
#                   'Engine.Type',
#                   'Number.of.Engines',
#                   'Aircraft.Category',
#                    'Aircraft.damage',
#                     'Injury.Severity',
#                   'Total.Fatal.Injuries',
#                   'Total.Serious.Injuries',
#                   'Total.Minor.Injuries',
#                   'Total.Uninjured',
#                   ]



In [116]:
(round(df.isna().sum()/len(df),4)*100).sort_values(ascending=False)

Schedule                  85.85
Air.carrier               81.27
FAR.Description           63.97
Aircraft.Category         63.68
Longitude                 61.33
Latitude                  61.32
Airport.Code              43.60
Airport.Name              40.71
Broad.phase.of.flight     30.56
Publication.Date          15.49
Total.Serious.Injuries    14.07
Total.Minor.Injuries      13.42
Total.Fatal.Injuries      12.83
Engine.Type                7.98
Report.Status              7.18
Purpose.of.flight          6.97
Number.of.Engines          6.84
Total.Uninjured            6.65
Weather.Condition          5.05
Aircraft.damage            3.59
Registration.Number        1.55
Injury.Severity            1.12
Country                    0.25
Amateur.Built              0.11
Model                      0.10
Make                       0.07
Location                   0.06
Accident.Number            0.00
Event.Date                 0.00
Investigation.Type         0.00
dtype: float64

In [117]:
#get a copy of the dataframe 
data=df.copy()

#convert the date into proper datetime format
from datetime import datetime
data["Publication.Date"]=pd.to_datetime(data["Publication.Date"],errors="coerce")
data["Event.Date"]=pd.to_datetime(data['Event.Date'],errors="coerce")

C:\Users\Lenovo\AppData\Local\Temp\ipykernel_16628\1659456149.py:6: UserWarning: Parsing dates in %d-%m-%Y format when dayfirst=False (the default) was specified. Pass `dayfirst=True` or specify a format to silence this warning.
  data["Publication.Date"]=pd.to_datetime(data["Publication.Date"],errors="coerce")


In [118]:
# this cell contained filtered data specifically needed by the client,

filtered_data=data[
                   (data["Event.Date"]>=pd.to_datetime("1983-01-01")) &
                   (data["Amateur.Built"]=="No")&
                   (data["Aircraft.Category"]=="Airplane")
                   ]

In [119]:
filtered_data.head()

,Investigation.Type,Accident.Number,Event.Date,Location,Country,Latitude,Longitude,Airport.Code,Airport.Name,Injury.Severity,...,Purpose.of.flight,Air.carrier,Total.Fatal.Injuries,Total.Serious.Injuries,Total.Minor.Injuries,Total.Uninjured,Weather.Condition,Broad.phase.of.flight,Report.Status,Publication.Date
Event.Id,,,,,,,,,,,,,,,,,,,,,
20001214X42478,Incident,LAX83IA149B,1983-03-18,"LOS ANGELES, CA",United States,NaN,NaN,LAX,LOS ANGELES INTL,Incident,...,Unknown,NaN,NaN,NaN,NaN,588.0,VMC,Standing,Probable Cause,2014-12-04
20001214X42478,Incident,LAX83IA149A,1983-03-18,"LOS ANGELES, CA",United States,NaN,NaN,LAX,LOS ANGELES INTL,Incident,...,NaN,"Singapore Airlines, Ltd.",NaN,NaN,NaN,588.0,VMC,Taxi,Probable Cause,2014-12-04
20001214X42331,Accident,ATL83FA140,1983-03-20,"CROSSVILLE, TN",United States,NaN,NaN,NaN,NaN,Fatal(1),...,Personal,NaN,1.0,1.0,NaN,NaN,IMC,Cruise,Probable Cause,2011-05-02
20001214X42672,Accident,FTW83LA177,1983-04-02,"MCKINNEY, TX",United States,NaN,NaN,TX05,AERO COUNTRY,Fatal(1),...,Skydiving,NaN,1.0,NaN,NaN,4.0,VMC,Standing,Probable Cause,2016-10-17
20001214X44248,Incident,MIA83IA210,1983-08-21,"NORFOLK, VA",United States,NaN,NaN,NaN,NaN,Incident,...,Unknown,NaN,NaN,NaN,NaN,289.0,VMC,Cruise,Probable Cause,2016-02-01


1. The data above is skewed,,decided to use median to remove the null values.

In [120]:
#get the median for injuries then use it to replace the null values 
med_fatal=filtered_data['Total.Fatal.Injuries'].median()
med_Uninjured=filtered_data['Total.Uninjured'].median()
med_sereous=filtered_data["Total.Serious.Injuries"].median()
med_minor=filtered_data["Total.Minor.Injuries"].median()

In [121]:
#lets fill the null values using median above 
filtered_data['Total.Fatal.Injuries'].fillna(med_fatal, inplace=True)
filtered_data['Total.Uninjured'].fillna(med_Uninjured,inplace=True)
filtered_data["Total.Serious.Injuries"].fillna(med_sereous,inplace=True)
filtered_data["Total.Minor.Injuries"].fillna(med_minor,inplace=True)



C:\Users\Lenovo\AppData\Local\Temp\ipykernel_16628\3786113135.py:2: ChainedAssignmentError: A value is being set on a copy of a DataFrame or Series through chained assignment using an inplace method.
Such inplace method never works to update the original DataFrame or Series, because the intermediate object on which we are setting values always behaves as a copy (due to Copy-on-Write).

For example, when doing 'df[col].method(value, inplace=True)', try using 'df.method({col: value}, inplace=True)' instead, to perform the operation inplace on the original object, or try to avoid an inplace operation using 'df[col] = df[col].method(value)'.

See the documentation for a more detailed explanation: https://pandas.pydata.org/pandas-docs/stable/user_guide/copy_on_write.html
  filtered_data['Total.Fatal.Injuries'].fillna(med_fatal, inplace=True)
C:\Users\Lenovo\AppData\Local\Temp\ipykernel_16628\3786113135.py:3: ChainedAssignmentError: A value is being set on a copy of a DataFrame or Series thr

Event.Id
20001214X42478    0.0
20001214X42478    0.0
20001214X42331    0.0
20001214X42672    0.0
20001214X44248    0.0
                 ... 
20221213106455    0.0
20221215106463    0.0
20221219106475    0.0
20221219106470    0.0
20221227106497    0.0
Name: Total.Minor.Injuries, Length: 21447, dtype: float64

### Cleaning and constructing Key Measurables

Injuries and robustness to destruction are a key interest point for the client. Clean and impute relevant columns and then create derived fields that best quantifies what the client wishes to track. **Use commenting or markdown to explain any cleaning assumptions as well as any derived columns you create.**

**Construct metric for fatal/serious injuries**

*Hint:* Estimate the total number of passengers on each flight. The likelihood of serious / fatal injury can be estimated as a fraction from this.

**Aircraft.Damage**
- identify and execute any cleaning tasks
- construct a derived column tracking whether an aircraft was destroyed or not.

### Investigate the *Make* column
- Identify cleaning tasks here
- List cleaning tasks clearly in markdown
- Execute the cleaning tasks
- For your analysis, keep Makes with a reasonable number (you can put the threshold at 50 though lower could work as well)

### Inspect Model column
- Get rid of any NaNs.
- Inspect the column and counts for each model/make. Are model labels unique to each make?
- If not, create a derived column that is a unique identifier for a given plane type.

### Cleaning other columns
- there are other columns containing data that might be related to the outcome of an accident. We list a few here:
- Engine.Type
- Weather.Condition
- Number.of.Engines
- Purpose.of.flight
- Broad.phase.of.flight

Inspect and identify potential cleaning tasks in each of the above columns. Execute those cleaning tasks. 

**Note**: You do not necessarily need to impute or drop NaNs here.

### Column Removal
- inspect the dataframe and drop any columns that have too many NaNs

### Save DataFrame to csv
- its generally useful to save data to file/server after its in a sufficiently cleaned or intermediate state
- the data can then be loaded directly in another notebook for further analysis
- this helps keep your notebooks and workflow readable, clean and modularized